# One Row Per Event

**Version**: 1

**Last modified**: 2026-10-12

**Compatible with opengate-gate-tree version:** 0.9.0

**Author**: Mateusz Jakub Bała

## About

The end of an analysis is usually a table: one row per event that passed, and
in its columns the values the next step needs - the energies and positions of
the hits, the kind of each gamma. `extract` gives that table, naming every
column after the track and hit it describes: `t1h2_edep` is the energy
deposited by the second hit of the first track.

Events seldom keep the same number of tracks, nor tracks of hits, while a table
has one width. This notebook builds tables, shows how wide they are and what
does not fit, and reads the columns of one hit back as a vector - to measure
how far apart the two annihilation gammas of an event first interacted.

## What will you learn from this notebook

- extract a table of the variables asked for, and read its column names
- know how wide a table is, what is left out when events differ, and how the
  log says so
- narrow a table with `max_tracks` and `max_hits`, and give an empty table its
  columns
- read the columns of one hit as a vector with the `gate` namespace

## Tutorial

Every step below is a line of text, the code that does what the line says, and
a word about the answer where the answer needs one.

In [1]:
from tools import environment, paths

print(environment.describe())

opengate-gate-tree: 0.8.0
python: 3.14.5
run on: 2026-10-12


The header of this notebook states the version it was written for; the line
above states the version it just ran against. If they differ, the notebook may
still be right — but nobody has checked it against what is installed.

The example file carried in the repository, described in
[`../data/README.md`](../data/README.md): 500 hits of a scene with a
`PositroniumSource` configured with every channel at once, beside a
back-to-back source. Nothing is written, so no output directory is needed.

In [2]:
import logging

from opengate_gate_tree import (
    DecayType,
    EventTrackHit,
    EventTrackHitAnalysis,
    EventTrackHitField,
    GammaType,
    GateTree,
    Process,
    SourceType,
    read_tree,
)

Analysis = EventTrackHitAnalysis
Field = EventTrackHitField
data = read_tree(paths.input_example("positronium-source-all-variants.root"), GateTree.HITS)
representation = EventTrackHit(data)
representation

EventTrackHit(events=115, tracks=311, hits=500)

### The Columns Of A Table

The first example of the analysis concept, closed with its count, and the
energy and position of every hit:

In [3]:
pps_pairs = (
    Analysis()
    .tracks(2)
    .gamma_type(GammaType.ANNIHILATION)
    .decay_type(DecayType.STANDARD)
    .source_type(SourceType.PARA_POSITRONIUM)
    .hits(eq=2)
    .hit(1)
    .process(Process.COMPTON)
    .hit(2)
    .process(Process.PHOTOELECTRIC)
    .tracks(2)
)
table = pps_pairs.extract(representation, [Field.EDEP, Field.POS_X, Field.POS_Y, Field.POS_Z])
list(table.columns)

['runID',
 'eventID',
 't1h1_edep',
 't1h2_edep',
 't2h1_edep',
 't2h2_edep',
 't1h1_posX',
 't1h2_posX',
 't2h1_posX',
 't2h2_posX',
 't1h1_posY',
 't1h2_posY',
 't2h1_posY',
 't2h2_posY',
 't1h1_posZ',
 't1h2_posZ',
 't2h1_posZ',
 't2h2_posZ']

`runID` and `eventID` come first, since an event is the pair of both. Then the
variables in the order of the list, and within each the tracks and the hits:
all the energies, then all the `posX`, and so on. A variable of events would
have one column, and one of tracks one per track, `t1_<name>`:

In [4]:
pps_pairs.extract(representation, [Field.SOURCE_TYPE, Field.TRACK_ID, Field.EDEP])

,runID,eventID,sourceType,t1_trackID,t2_trackID,t1h1_edep,t1h2_edep,t2h1_edep,t2h2_edep
0,0,0,2,1,2,0.220403,0.290596,0.094538,0.416461


The two deposits of each gamma add up to 511 keV: a gamma that scattered once
and was then absorbed left all its energy in the detector.

### How Wide A Table Is

An analysis that does not fix its counts keeps events of different sizes. The
table is as wide as the narrowest of them allows, so that no cell is empty, and
what does not fit is left out - and counted in the log, once logging is on:

In [5]:
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(message)s", force=True)

annihilation = Analysis().tracks().gamma_type(GammaType.ANNIHILATION)
wide = annihilation.extract(representation, [Field.TRACK_ID, Field.EDEP])
wide.shape, list(wide.columns)

WARNING A table of 2 track(s) per event and 1 hit(s) per track left out 51 track(s) of 51 event(s) and 103 hit(s) of 75 track(s); end the analysis with tracks(n) and hits(n) to keep a fixed number of each.


((97, 6),
 ['runID', 'eventID', 't1_trackID', 't2_trackID', 't1h1_edep', 't2h1_edep'])

Every event kept at least two annihilation gammas and every gamma at least one
hit, so the table holds two tracks and one hit of each. The third annihilation
gamma of every ortho-positronium decay, and every hit after the first, were
left out, as the log says. An analysis that ends with fixed counts - as the
first one does - loses nothing.

`max_tracks` and `max_hits` narrow a table further, and refuse to widen it,
which would leave cells empty:

In [6]:
narrow = annihilation.extract(representation, [Field.EDEP], max_tracks=1)
list(narrow.columns)

WARNING A table of 1 track(s) per event and 1 hit(s) per track left out 148 track(s) of 97 event(s) and 55 hit(s) of 38 track(s); end the analysis with tracks(n) and hits(n) to keep a fixed number of each.


['runID', 'eventID', 't1h1_edep']

In [7]:
try:
    annihilation.extract(representation, [Field.EDEP], max_tracks=3)
except ValueError as error:
    print(error)

A table of 3 tracks per event would leave cells empty: at least one event of the result kept only 2. Ask for 2 or fewer.


### A Table Of Nothing

When no event passes, there is no event to take a width from. `max_tracks` and
`max_hits` then give the width of the empty table, so code that reads its
columns finds them whether anything passed or not:

In [8]:
nothing = Analysis().tracks(9)
empty = nothing.extract(representation, [Field.EDEP], max_tracks=2, max_hits=1)
empty.shape, list(empty.columns)

((0, 4), ['runID', 'eventID', 't1h1_edep', 't2h1_edep'])

### Columns Read As Vectors

The columns keep the names of the branches, so the three coordinates of one hit
are a vector to the `gate` namespace. How far apart the two annihilation gammas
of an event first interacted:

In [9]:
pairs = Analysis().tracks().gamma_type(GammaType.ANNIHILATION).tracks(2)
positions = pairs.extract(representation, [Field.POS_X, Field.POS_Y, Field.POS_Z])

first = positions.gate.vector("t1h1_posX", "t1h1_posY", "t1h1_posZ")
second = positions.gate.vector("t2h1_posX", "t2h1_posY", "t2h1_posZ")
distance = (second - first).norm()
distance.describe()[["count", "min", "mean", "max"]]

WARNING A table of 2 track(s) per event and 1 hit(s) per track left out 0 track(s) of 0 event(s) and 70 hit(s) of 48 track(s); end the analysis with tracks(n) and hits(n) to keep a fixed number of each.


count     46.000000
min      401.935617
mean     416.216130
max      448.032826
Name: norm, dtype: float64

Never less than 400 mm. The two gammas of an annihilation fly apart back to
back, and the detector is a shell with an inner radius of 200 mm, so their
first interactions lie on opposite sides of it - at least a diameter apart.

## Summary

`extract` gives one row per event, with `runID` and `eventID` first and a column
per variable, track and hit, named `t{i}h{j}_<branch>`. The table is as wide as
the narrowest event and track allow, says in the log what it left out, can be
narrowed or given a width when empty, and its columns read back as vectors.

## Bibliography

- [Analysis Of Events, Tracks And Hits](https://opengate-gate-tree.readthedocs.io/en/latest/guide/analysis.html)
  — the table, its columns and its width
- [Events, Tracks And Hits](https://opengate-gate-tree.readthedocs.io/en/latest/guide/event-track-hit.html)
  — the representation an analysis is performed on, and the order it counts in
- [PositroniumSource Data](https://opengate-gate-tree.readthedocs.io/en/latest/guide/positronium.html)
  — what `sourceType`, `decayType` and `gammaType` mean

## Changelog

- v1: written.